# CDT-III 2-Stage VCE: Gradient Analysis + TRS (NB2)
## Dual Gradients, Translational Regulation Score, Side Effects & Resistance

**Purpose**: RNA vs Protein gradient comparison. TRS for post-translational regulation discovery.
Side effect prediction and resistance mechanism analysis for CD52/Alemtuzumab.

**CDT-III Unique**: TRS, side effect maps, resistance genes — none possible with CDT-II.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List
import numpy as np
import h5py
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import json
import gc

print(f"PyTorch: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ============================================================
# Path Configuration
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")
OUTPUT_BASE = Path("/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2")

CELLLEVEL_WITH_ADT_PATH = DRIVE_BASE / "morris_celllevel_effects_with_adt.h5"
TSS_ENFORMER_PATH = DRIVE_BASE / "morris_28genes_enformer.h5"
BEST_MODEL_PATH = OUTPUT_BASE / "cdt_2stage_vce_best.pt"
ADT_MAPPING_PATH = "/content/drive/MyDrive/cdt_data/adt_mapping.csv"

FIGURES_DIR = OUTPUT_BASE / "gradient_trs_figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("File check:")
for name, path in [
    ("Cell-level + ADT", CELLLEVEL_WITH_ADT_PATH),
    ("TSS Enformer", TSS_ENFORMER_PATH),
    ("Best model", BEST_MODEL_PATH),
]:
    exists = path.exists()
    print(f"  {'✓' if exists else '✗'} {name}: {path}")

In [ ]:
# ============================================================
# CDT-III 2-Stage VCE Model Definition
# (Copy from CDT_Morris_2StageVCE_v2_Training.ipynb cells 12-16)
# ============================================================

@dataclass
class CDT2StageVCEConfig:
    dna_dim: int = 3072
    dna_seq_len: int = 896
    n_genes: int = 2361
    n_proteins: int = 189
    hidden_dim: int = 512
    nhead: int = 8
    dropout: float = 0.3
    protein_dropout: float = 0.5
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1
    protein_self_attn_layers: int = 1


class RawExpressionEncoder(nn.Module):
    def __init__(self, n_features, hidden_dim, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        self.feature_embedding = nn.Embedding(n_features, hidden_dim)
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout))
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout))

    def forward(self, expression):
        B = expression.size(0)
        feat_ids = torch.arange(self.n_features, device=expression.device)
        feat_emb = self.feature_embedding(feat_ids).unsqueeze(0).expand(B, -1, -1)
        expr_emb = self.expr_projector(expression.unsqueeze(-1))
        return self.combine(torch.cat([feat_emb, expr_emb], dim=-1))


class SequenceProjector(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.norm(self.linear(x)))


class FlashSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, return_attn=False):
        B, S, _ = x.shape
        Q = self.q_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
            attn_weights = None
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        if return_attn:
            return x, attn_weights
        return x


class FlashCrossAttentionBlock(nn.Module):
    def __init__(self, d_model, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value, return_attn=False):
        B, QL, _ = query.shape
        KL = key_value.shape[1]
        Q = self.q_proj(query).view(B, QL, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(B, KL, self.nhead, self.head_dim).transpose(1, 2)
        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V, dropout_p=self.dropout_p if self.training else 0.0)
            attn_weights = None
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, QL, self.d_model)
        attn_out = self.out_proj(attn_out)
        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))
        if return_attn:
            return x, attn_weights
        return x


class VirtualCellEmbedderDNARNA(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)
        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B, S = key_value.size(0), key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        dna_pooled = self._attention_pool(
            self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj)
        rna_pooled = self._attention_pool(
            self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj)
        return self.fusion(torch.cat([dna_pooled, rna_pooled], dim=-1))


class VirtualCellEmbedderProtein(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead
        self.protein_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.protein_q_proj = nn.Linear(d_model, d_model)
        self.protein_k_proj = nn.Linear(d_model, d_model)
        self.protein_v_proj = nn.Linear(d_model, d_model)
        self.protein_out_proj = nn.Linear(d_model, d_model)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model))

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        B, S = key_value.size(0), key_value.size(1)
        query = query.expand(B, -1, -1)
        Q = q_proj(query).view(B, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(B, S, self.nhead, self.head_dim).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, cell_emb_rna, protein_encoded):
        protein_pooled = self._attention_pool(
            self.protein_query, protein_encoded,
            self.protein_q_proj, self.protein_k_proj,
            self.protein_v_proj, self.protein_out_proj)
        return self.fusion(torch.cat([cell_emb_rna, protein_pooled], dim=-1))


class CDTTrimodal2StageModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        if config is None:
            config = CDT2StageVCEConfig()
        self.config = config
        self.dna_projector = SequenceProjector(config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)])
        self.rna_encoder = RawExpressionEncoder(config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)])
        self.dna_to_rna = FlashCrossAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
        self.vce_t = VirtualCellEmbedderDNARNA(config.hidden_dim, config.dropout)
        self.rna_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(),
            nn.Dropout(config.dropout), nn.Linear(config.hidden_dim, config.n_genes))
        self.protein_encoder = RawExpressionEncoder(
            config.n_proteins, config.hidden_dim, config.protein_dropout)
        self.protein_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.protein_dropout)
            for _ in range(config.protein_self_attn_layers)])
        self.rna_to_protein = FlashCrossAttentionBlock(
            config.hidden_dim, config.nhead, config.protein_dropout)
        self.vce_p = VirtualCellEmbedderProtein(config.hidden_dim, config.protein_dropout)
        self.protein_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim), nn.GELU(),
            nn.Dropout(config.protein_dropout),
            nn.Linear(config.hidden_dim, config.n_proteins))

    def forward(self, dna_emb, rna_expr, protein_expr, return_attention=False):
        attn_maps = {} if return_attention else None
        dna = self.dna_projector(dna_emb)
        rna = self.rna_encoder(rna_expr)
        protein = self.protein_encoder(protein_expr)
        for i, layer in enumerate(self.dna_self_attn_layers):
            if return_attention:
                dna, attn_w = layer(dna, return_attn=True)
                attn_maps[f'dna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                dna = layer(dna)
        for i, layer in enumerate(self.rna_self_attn_layers):
            if return_attention:
                rna, attn_w = layer(rna, return_attn=True)
                attn_maps[f'rna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                rna = layer(rna)
        for i, layer in enumerate(self.protein_self_attn_layers):
            if return_attention:
                protein, attn_w = layer(protein, return_attn=True)
                attn_maps[f'protein_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                protein = layer(protein)
        if return_attention:
            rna, attn_w = self.dna_to_rna(query=rna, key_value=dna, return_attn=True)
            attn_maps['dna_to_rna_cross'] = attn_w.detach().cpu()
        else:
            rna = self.dna_to_rna(query=rna, key_value=dna)
        if return_attention:
            protein, attn_w = self.rna_to_protein(query=protein, key_value=rna, return_attn=True)
            attn_maps['rna_to_protein_cross'] = attn_w.detach().cpu()
        else:
            protein = self.rna_to_protein(query=protein, key_value=rna)
        cell_emb_rna = self.vce_t(dna, rna)
        rna_pred = self.rna_task_layer(cell_emb_rna)
        cell_emb_protein = self.vce_p(cell_emb_rna, protein)
        protein_pred = self.protein_task_layer(cell_emb_protein)
        if return_attention:
            return rna_pred, protein_pred, attn_maps
        return rna_pred, protein_pred

    def get_num_params(self, trainable_only=False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())

print("CDTTrimodal2StageModel defined.")

In [ ]:
# ============================================================
# Load Data
# ============================================================
print("Loading integrated RNA + ADT data...")
with h5py.File(CELLLEVEL_WITH_ADT_PATH, 'r') as f:
    rna_log2fc = f['log2fc'][:]
    cell_expr = f['cell_expr'][:]
    target_gene_idx = f['target_gene_idx'][:]
    cdt_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names_cdt'][:]]
    target_gene_names = [g.decode() if isinstance(g, bytes) else g for g in f['target_gene_names'][:]]
    ntc_mean_expr = f['ntc_mean_expr'][:]
    train_genes = [g.decode() if isinstance(g, bytes) else g for g in f['train_genes'][:]]
    val_genes = [g.decode() if isinstance(g, bytes) else g for g in f['val_genes'][:]]
    protein_effect = f['protein_effect'][:]
    protein_dsb = f['protein_dsb'][:]
    protein_names = [g.decode() if isinstance(g, bytes) else g for g in f['protein_names'][:]]
    n_cells = f.attrs['n_cells']
    n_genes = f.attrs['n_genes']
    n_proteins = f.attrs['n_proteins']

    # Check if 'protein_matched_mask' exists, otherwise create a default all-True mask
    if 'protein_matched_mask' in f:
        protein_matched_mask = f['protein_matched_mask'][:]
    else:
        print("Warning: 'protein_matched_mask' not found in HDF5. Creating an all-True mask.")
        protein_matched_mask = np.ones(n_cells, dtype=bool)

print(f"RNA: {n_cells} cells, {n_genes} genes, log2FC {rna_log2fc.shape}")
print(f"Protein: {n_proteins} proteins, effect {protein_effect.shape}")
print(f"Cells with protein: {protein_matched_mask.sum()}/{n_cells}")
print(f"Train genes ({len(train_genes)}): {train_genes[:5]}...")
print(f"Val genes ({len(val_genes)}): {val_genes}")

# Enformer
print("\nLoading Enformer embeddings...")
with h5py.File(TSS_ENFORMER_PATH, 'r') as f:
    enformer_emb = f['embeddings'][:]
    enformer_genes = [g.decode() if isinstance(g, bytes) else g for g in f['gene_names'][:]]
print(f"Enformer: {enformer_emb.shape}")

target_to_enformer = {}
for i, gene in enumerate(target_gene_names):
    if gene in enformer_genes:
        target_to_enformer[i] = enformer_genes.index(gene)
print(f"TSS matched: {len(target_to_enformer)}/{len(target_gene_names)}")

# ADT mapping
try:
    adt_mapping = pd.read_csv(ADT_MAPPING_PATH)
    adt_mapping = adt_mapping[~adt_mapping['morris_adt_id'].str.startswith('ADT-CTL')]
    adt_to_gene = dict(zip(adt_mapping['morris_adt_id'], adt_mapping['gene_symbol']))
    adt_to_protein = dict(zip(adt_mapping['morris_adt_id'], adt_mapping['protein_name']))
    print(f"ADT mapping: {len(adt_to_gene)} proteins mapped")
except:
    print("ADT mapping not found, using protein_names directly")
    adt_to_gene = {}
    adt_to_protein = {}

# Index maps
gene_to_cdt_idx = {g: i for i, g in enumerate(cdt_genes)}
gene_name_to_target_idx = {n: i for i, n in enumerate(target_gene_names)}

# ADT screening: expressed proteins
mean_dsb = protein_dsb.mean(axis=0)
std_dsb = protein_dsb.std(axis=0)
expressed_mask = (mean_dsb > 0.5) & (std_dsb > 0.1)
expressed_indices = np.where(expressed_mask)[0]
n_expressed = expressed_mask.sum()
print(f"\nExpressed proteins: {n_expressed}/{n_proteins}")

In [ ]:
import os
output_dir = "/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2/"
if os.path.exists(output_dir):
    for f in sorted(os.listdir(output_dir)):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f} ({size/1e6:.1f} MB)")
else:
    print("Directory not found!")
    # Check parent
    parent = "/content/drive/MyDrive/cdt_outputs/"
    if os.path.exists(parent):
        print("Available dirs:")
        for d in sorted(os.listdir(parent)):
            print(f"  {d}")


In [ ]:
# ============================================================
# Load Best Model Checkpoint
# ============================================================
print("Loading best model checkpoint...")
config = CDT2StageVCEConfig(n_genes=len(cdt_genes), n_proteins=len(protein_names))
model = CDTTrimodal2StageModel(config).to(device)

ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
if 'model_state_dict' in ckpt:
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded from checkpoint with keys: {list(ckpt.keys())}")
    if 'val_rna_r' in ckpt:
        print(f"  Val RNA r: {ckpt['val_rna_r']:.4f}")
    if 'val_protein_r' in ckpt:
        print(f"  Val Protein r: {ckpt['val_protein_r']:.4f}")
    if 'epoch' in ckpt:
        print(f"  Epoch: {ckpt['epoch']}")
else:
    model.load_state_dict(ckpt)
    print("Loaded raw state dict")

model.eval()
print(f"Parameters: {model.get_num_params():,}")

In [ ]:
# ============================================================
# A. Dual Gradient Computation
# ============================================================

def compute_dual_gradients(model, cell_indices, enformer_emb, cell_expr,
                           protein_dsb, target_gene_idx, target_to_enformer,
                           target_cdt_idx, target_protein_idx, device, max_cells=50):
    """Compute RNA and Protein gradients w.r.t. RNA input, averaged over cells.

    Returns:
        rna_grad_avg: [n_genes] — d(RNA_pred[target_gene]) / d(rna_input)
        prot_grad_avg: [n_genes] — d(Protein_pred[target_protein]) / d(rna_input)
    """
    rna_grads, prot_grads = [], []

    for ci in cell_indices[:max_cells]:
        enf_idx = target_to_enformer[target_gene_idx[ci]]
        dna_input = torch.from_numpy(
            enformer_emb[enf_idx:enf_idx+1].astype(np.float32)).to(device)
        rna_input = torch.from_numpy(
            cell_expr[ci:ci+1].astype(np.float32)).to(device)
        rna_input.requires_grad_(True)
        prot_input = torch.from_numpy(
            protein_dsb[ci:ci+1].astype(np.float32)).to(device)

        rna_pred, prot_pred = model(dna_input, rna_input, prot_input)

        # RNA gradient
        model.zero_grad()
        rna_pred[0, target_cdt_idx].backward(retain_graph=True)
        rna_grad = rna_input.grad.detach().cpu().numpy().flatten()
        rna_grads.append(rna_grad)

        # Protein gradient
        rna_input.grad = None
        prot_pred[0, target_protein_idx].backward()
        prot_grad = rna_input.grad.detach().cpu().numpy().flatten()
        prot_grads.append(prot_grad)

    return np.mean(rna_grads, axis=0), np.mean(prot_grads, axis=0)


def compute_protein_gradients_all(model, cell_indices, enformer_emb, cell_expr,
                                   protein_dsb, target_gene_idx, target_to_enformer,
                                   target_cdt_idx, n_proteins, device, max_cells=30):
    """Compute d(Protein_pred[j]) / d(rna_input[target_gene]) for ALL proteins.

    Returns:
        grad_matrix: [n_proteins] — gradient of each protein w.r.t. target gene expression
    """
    all_grads = []

    for ci in cell_indices[:max_cells]:
        enf_idx = target_to_enformer[target_gene_idx[ci]]
        dna_input = torch.from_numpy(
            enformer_emb[enf_idx:enf_idx+1].astype(np.float32)).to(device)
        rna_input = torch.from_numpy(
            cell_expr[ci:ci+1].astype(np.float32)).to(device)
        rna_input.requires_grad_(True)
        prot_input = torch.from_numpy(
            protein_dsb[ci:ci+1].astype(np.float32)).to(device)

        rna_pred, prot_pred = model(dna_input, rna_input, prot_input)

        # Gradient of each protein output w.r.t. target gene input
        grads_per_protein = np.zeros(n_proteins)
        for j in range(n_proteins):
            model.zero_grad()
            if rna_input.grad is not None:
                rna_input.grad = None
            prot_pred[0, j].backward(retain_graph=True)
            grads_per_protein[j] = rna_input.grad[0, target_cdt_idx].item()

        all_grads.append(grads_per_protein)

    return np.mean(all_grads, axis=0)

print("Gradient functions defined.")

In [ ]:
# ============================================================
# B. RNA Gradient Validation (CDT-II Reproduction)
# ============================================================
# Validate: |RNA gradient| vs |experimental log2FC| Pearson r
# Target: r ≈ 0.83 (matching CDT-II)

from numpy.linalg import norm

gradient_results = {}

for gene_name in val_genes:
    if gene_name not in gene_name_to_target_idx:
        continue
    g_target_idx = gene_name_to_target_idx[gene_name]
    if g_target_idx not in target_to_enformer:
        continue
    g_cdt_idx = gene_to_cdt_idx.get(gene_name)
    if g_cdt_idx is None:
        continue

    # Find matching ADT protein
    g_adt_idx = None
    for i, pname in enumerate(protein_names):
        if adt_to_gene.get(pname) == gene_name:
            g_adt_idx = i
            break

    gene_cells = [i for i in range(n_cells)
                  if target_gene_idx[i] == g_target_idx
                  and protein_matched_mask[i]]
    if len(gene_cells) < 10:
        continue

    # Default protein target if no matching ADT
    prot_idx = g_adt_idx if g_adt_idx is not None else 0

    rna_grad, prot_grad = compute_dual_gradients(
        model, gene_cells, enformer_emb, cell_expr, protein_dsb,
        target_gene_idx, target_to_enformer,
        g_cdt_idx, prot_idx, device, max_cells=50)

    # RNA gradient vs experimental log2FC
    mean_log2fc = np.mean([rna_log2fc[ci] for ci in gene_cells], axis=0)
    grad_vs_exp_r = pearsonr(np.abs(rna_grad), np.abs(mean_log2fc))[0]

    # TRS
    cos_sim = np.dot(rna_grad, prot_grad) / (norm(rna_grad) * norm(prot_grad) + 1e-8)
    trs = 1 - cos_sim

    gradient_results[gene_name] = {
        'rna_grad': rna_grad,
        'prot_grad': prot_grad,
        'grad_vs_exp_r': grad_vs_exp_r,
        'trs': trs,
        'cos_sim': cos_sim,
        'n_cells': len(gene_cells),
        'has_matching_adt': g_adt_idx is not None,
        'protein_idx': prot_idx,
    }
    print(f"  {gene_name}: gradient r={grad_vs_exp_r:.4f}, TRS={trs:.4f} "
          f"({len(gene_cells)} cells, ADT={'✓' if g_adt_idx is not None else '✗'})")

print(f"\nMean gradient r: {np.mean([r['grad_vs_exp_r'] for r in gradient_results.values()]):.4f}")
print(f"Mean TRS: {np.mean([r['trs'] for r in gradient_results.values()]):.4f}")

In [ ]:
# ============================================================
# C. TRS Visualization & Ranking
# ============================================================

trs_df = pd.DataFrame([
    {'gene': g, 'trs': r['trs'], 'cos_sim': r['cos_sim'],
     'grad_vs_exp_r': r['grad_vs_exp_r'], 'n_cells': r['n_cells'],
     'has_adt': r['has_matching_adt']}
    for g, r in gradient_results.items()
]).sort_values('trs', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: TRS bar chart
colors = ['#e74c3c' if g == 'TFRC' else '#3498db' for g in trs_df['gene']]
axes[0].barh(range(len(trs_df)), trs_df['trs'].values, color=colors)
axes[0].set_yticks(range(len(trs_df)))
axes[0].set_yticklabels(trs_df['gene'].values)
axes[0].invert_yaxis()
axes[0].set_xlabel('TRS (Translational Regulation Score)')
axes[0].set_title('A. TRS Ranking\nHigh = RNA-Protein divergence', fontsize=13)
axes[0].axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)

# Panel B: RNA gradient vs experimental log2FC
axes[1].barh(range(len(trs_df)), trs_df['grad_vs_exp_r'].values, color='#4CAF50')
axes[1].set_yticks(range(len(trs_df)))
axes[1].set_yticklabels(trs_df['gene'].values)
axes[1].invert_yaxis()
axes[1].set_xlabel('Pearson r (|gradient| vs |log2FC|)')
axes[1].set_title('B. RNA Gradient Validation\nTarget: r ≈ 0.83', fontsize=13)

# Panel C: RNA gradient scatter for best gene
best_g = trs_df.iloc[0]['gene'] if len(trs_df) > 0 else val_genes[0]
if best_g in gradient_results:
    rna_g = np.abs(gradient_results[best_g]['rna_grad'])
    prot_g = np.abs(gradient_results[best_g]['prot_grad'])
    axes[2].scatter(rna_g, prot_g, alpha=0.2, s=5, c='gray')
    axes[2].set_xlabel('|RNA gradient|')
    axes[2].set_ylabel('|Protein gradient|')
    max_val = max(rna_g.max(), prot_g.max())
    axes[2].plot([0, max_val], [0, max_val], 'k--', alpha=0.3)
    axes[2].set_title(f'C. {best_g}: RNA vs Protein Gradient', fontsize=13)

plt.suptitle('CDT-III: Dual Gradient Analysis & TRS', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gradient_trs_overview.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# D. Side Effect Prediction: CD52 KD → All Protein Changes
# ============================================================
# CD52 KD simulates Alemtuzumab treatment.
# Compute d(Protein_pred[j]) / d(rna_input[CD52]) for all 189 proteins.

CD52_CDT_IDX = gene_to_cdt_idx.get('CD52')
CD52_TARGET_IDX = gene_name_to_target_idx.get('CD52')

if CD52_CDT_IDX is not None and CD52_TARGET_IDX is not None:
    cd52_cells = [i for i in range(n_cells)
                  if target_gene_idx[i] == CD52_TARGET_IDX
                  and CD52_TARGET_IDX in target_to_enformer
                  and protein_matched_mask[i]]
    print(f"CD52 cells: {len(cd52_cells)}")

    # Compute gradient of each protein w.r.t. CD52 expression
    print("Computing CD52 → all proteins gradient (this may take a few minutes)...")
    cd52_protein_grads = compute_protein_gradients_all(
        model, cd52_cells, enformer_emb, cell_expr, protein_dsb,
        target_gene_idx, target_to_enformer,
        CD52_CDT_IDX, n_proteins, device, max_cells=30)

    # Rank proteins by absolute gradient (most affected by CD52 KD)
    abs_grads = np.abs(cd52_protein_grads)
    ranked_idx = np.argsort(abs_grads)[::-1]

    print(f"\nTop 20 proteins most affected by CD52 KD (Alemtuzumab):")
    side_effect_data = []
    for rank, idx in enumerate(ranked_idx[:30]):
        prot_label = adt_to_protein.get(protein_names[idx], protein_names[idx])
        gene_label = adt_to_gene.get(protein_names[idx], '')
        grad_val = cd52_protein_grads[idx]
        direction = '↓' if grad_val < 0 else '↑'
        is_expressed = expressed_mask[idx]
        side_effect_data.append({
            'rank': rank + 1,
            'protein': prot_label,
            'gene': gene_label,
            'protein_id': protein_names[idx],
            'gradient': grad_val,
            'abs_gradient': abs_grads[idx],
            'direction': direction,
            'expressed': is_expressed,
        })
        if rank < 20:
            expr_tag = '' if is_expressed else ' (low expr)'
            print(f"  {rank+1:2d}. {prot_label:<30s} ({gene_label:<8s}) "
                  f"grad={grad_val:+.6f} {direction}{expr_tag}")

    side_effect_df = pd.DataFrame(side_effect_data)

    # Known Alemtuzumab side effects — check for relevant proteins
    immune_markers = {
        'CD3': 'T cell (infection risk)',
        'CD4': 'Helper T cell (infection risk)',
        'CD8': 'Cytotoxic T (infection risk)',
        'CD19': 'B cell (infection risk)',
        'CD52': 'Drug target (self)',
        'CD55': 'Complement regulator (CDC escape)',
        'CD59': 'Complement regulator (CDC escape)',
    }
    print(f"\n--- Known Alemtuzumab-related markers ---")
    for marker, role in immune_markers.items():
        for idx, pname in enumerate(protein_names):
            gene = adt_to_gene.get(pname, '')
            if gene == marker or marker in pname:
                grad_val = cd52_protein_grads[idx]
                print(f"  {marker}: {adt_to_protein.get(pname, pname)} "
                      f"grad={grad_val:+.6f} ({role})")
else:
    print("CD52 not found in gene list.")
    cd52_protein_grads = None

In [ ]:
# ============================================================
# E. Resistance Mechanism Analysis
# ============================================================
# Resistance = mRNA changes but protein is maintained (post-translational compensation)
# Identify genes where RNA gradient >> 0 but Protein gradient ≈ 0

if 'CD52' in gradient_results:
    rna_g = gradient_results['CD52']['rna_grad']
    prot_g = gradient_results['CD52']['prot_grad']

    rna_abs = np.abs(rna_g)
    prot_abs = np.abs(prot_g)

    # Resistance score: high RNA gradient, low Protein gradient
    # (mRNA changes but protein is stabilized)
    resistance_score = rna_abs / (prot_abs + 1e-8)

    # Filter: only consider genes with meaningful RNA gradient
    meaningful_mask = rna_abs > np.percentile(rna_abs, 75)
    resistance_candidates = np.where(meaningful_mask)[0]
    resistance_ranked = resistance_candidates[
        np.argsort(resistance_score[resistance_candidates])[::-1]]

    print("Top 20 Resistance Candidate Genes (high RNA, low Protein gradient):")
    print(f"{'Rank':>4} {'Gene':<12} {'RNA grad':>10} {'Prot grad':>10} {'Ratio':>10}")
    print("-" * 50)
    resistance_data = []
    for rank, idx in enumerate(resistance_ranked[:20]):
        gene_name = cdt_genes[idx]
        resistance_data.append({
            'rank': rank + 1,
            'gene': gene_name,
            'rna_grad': rna_g[idx],
            'prot_grad': prot_g[idx],
            'resistance_score': resistance_score[idx],
        })
        print(f"  {rank+1:2d}. {gene_name:<12s} {rna_g[idx]:+.6f} "
              f"{prot_g[idx]:+.6f} {resistance_score[idx]:.1f}")

    resistance_df = pd.DataFrame(resistance_data)

    # Check known resistance genes
    known_resistance = ['CD55', 'CD59', 'PIGF', 'PIGA', 'PIGB']
    print(f"\n--- Known Alemtuzumab resistance pathway genes ---")
    for gene in known_resistance:
        if gene in gene_to_cdt_idx:
            idx = gene_to_cdt_idx[gene]
            print(f"  {gene}: RNA grad={rna_g[idx]:+.6f}, "
                  f"Prot grad={prot_g[idx]:+.6f}, "
                  f"ratio={resistance_score[idx]:.1f}")
        else:
            print(f"  {gene}: not in CDT gene list")
else:
    print("CD52 gradient data not available.")
    resistance_df = None

In [ ]:
# ============================================================
# F. Jacobian Heatmaps: RNA vs Protein (CD52, GFI1B)
# ============================================================

def compute_jacobian_row(model, cell_idx, enformer_emb, cell_expr, protein_dsb,
                         target_gene_idx, target_to_enformer, output_indices,
                         output_type, device):
    """Compute one row of the Jacobian: d(output[i]) / d(rna_input) for selected outputs."""
    enf_idx = target_to_enformer[target_gene_idx[cell_idx]]
    dna_t = torch.from_numpy(enformer_emb[enf_idx:enf_idx+1].astype(np.float32)).to(device)
    rna_input = torch.from_numpy(cell_expr[cell_idx:cell_idx+1].astype(np.float32)).to(device)
    rna_input.requires_grad_(True)
    prot_t = torch.from_numpy(protein_dsb[cell_idx:cell_idx+1].astype(np.float32)).to(device)

    rna_pred, prot_pred = model(dna_t, rna_input, prot_t)
    output = rna_pred if output_type == 'rna' else prot_pred

    jacobian_rows = []
    for oi in output_indices:
        model.zero_grad()
        if rna_input.grad is not None:
            rna_input.grad = None
        output[0, oi].backward(retain_graph=True)
        jacobian_rows.append(rna_input.grad[0].detach().cpu().numpy())

    return np.array(jacobian_rows)  # [n_outputs, n_genes]


for target_gene in ['CD52', 'GFI1B']:
    if target_gene not in gene_name_to_target_idx:
        print(f"{target_gene}: not found, skipping")
        continue
    g_target_idx = gene_name_to_target_idx[target_gene]
    if g_target_idx not in target_to_enformer:
        continue

    gene_cells = [i for i in range(n_cells)
                  if target_gene_idx[i] == g_target_idx
                  and protein_matched_mask[i]]
    if len(gene_cells) < 5:
        continue

    ci = gene_cells[0]

    # Select top affected RNA outputs
    mean_log2fc = np.mean([rna_log2fc[ci_] for ci_ in gene_cells[:50]], axis=0)
    top_rna_outputs = np.argsort(np.abs(mean_log2fc))[::-1][:50]

    # Select expressed protein outputs
    top_prot_outputs = expressed_indices[:min(30, len(expressed_indices))]

    # Compute RNA Jacobian
    rna_jac = compute_jacobian_row(
        model, ci, enformer_emb, cell_expr, protein_dsb,
        target_gene_idx, target_to_enformer,
        top_rna_outputs, 'rna', device)

    # Compute Protein Jacobian
    prot_jac = compute_jacobian_row(
        model, ci, enformer_emb, cell_expr, protein_dsb,
        target_gene_idx, target_to_enformer,
        top_prot_outputs, 'protein', device)

    # Select top input genes (union of RNA and Protein top gradients)
    rna_input_importance = np.abs(rna_jac).mean(axis=0)
    prot_input_importance = np.abs(prot_jac).mean(axis=0)
    combined = rna_input_importance + prot_input_importance
    top_input_genes = np.argsort(combined)[::-1][:40]

    # Plot side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    # RNA Jacobian
    rna_sub = np.abs(rna_jac[:, top_input_genes])
    rna_labels_y = [cdt_genes[i] for i in top_rna_outputs]
    rna_labels_x = [cdt_genes[i] for i in top_input_genes]
    im1 = axes[0].imshow(rna_sub, cmap='YlOrRd', aspect='auto')
    axes[0].set_title(f'{target_gene}: RNA Jacobian', fontsize=13)
    axes[0].set_xlabel('Input genes')
    axes[0].set_ylabel('Output genes (RNA)')
    plt.colorbar(im1, ax=axes[0], shrink=0.6)

    # Protein Jacobian
    prot_sub = np.abs(prot_jac[:, top_input_genes])
    prot_labels_y = [adt_to_protein.get(protein_names[i], protein_names[i])[:15]
                     for i in top_prot_outputs]
    im2 = axes[1].imshow(prot_sub, cmap='YlOrRd', aspect='auto')
    axes[1].set_title(f'{target_gene}: Protein Jacobian ★NEW', fontsize=13)
    axes[1].set_xlabel('Input genes')
    axes[1].set_ylabel('Output proteins')
    plt.colorbar(im2, ax=axes[1], shrink=0.6)

    plt.suptitle(f'{target_gene}: RNA vs Protein Jacobian Heatmap',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'jacobian_{target_gene}.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved: jacobian_{target_gene}.png")
    gc.collect()

## NTC Mean Gradient Validation

**Key question**: Can gradient analysis predict CD52 side effects using **only NTC mean** (no perturbation data)?

This validates the paper's claim that gradient analysis can screen novel drug targets without conducting perturbation experiments.
- Input: NTC mean RNA [2361] + NTC mean protein DSB [189] + CD52 Enformer DNA embedding
- No CD52 perturbation cell data is used
- Compare results with Cell 10 (which used actual CD52-perturbed cells)

In [ ]:
# ============================================================
# NTC Mean Gradient Analysis: CD52 Side Effect Prediction
# WITHOUT any CD52 perturbation data
# ============================================================

# --- Step 1: Compute NTC mean protein DSB ---
# Try to load from HDF5, otherwise compute from NTC cells
ntc_mean_protein = None
with h5py.File(CELLLEVEL_WITH_ADT_PATH, 'r') as f:
    if 'ntc_mean_protein' in f:
        ntc_mean_protein = f['ntc_mean_protein'][:]
        print(f'Loaded ntc_mean_protein from HDF5: {ntc_mean_protein.shape}')

if ntc_mean_protein is None:
    # Compute from NTC cells
    ntc_cell_indices = [i for i in range(n_cells)
                        if target_gene_names[target_gene_idx[i]] == 'non-targeting']
    if len(ntc_cell_indices) > 0:
        ntc_mean_protein = protein_dsb[ntc_cell_indices].mean(axis=0)
        print(f'Computed NTC mean protein from {len(ntc_cell_indices)} NTC cells')
    else:
        # Fallback: use overall mean
        ntc_mean_protein = protein_dsb.mean(axis=0)
        print('WARNING: NTC cells not found. Using overall protein mean as fallback.')

print(f'NTC mean RNA shape: {ntc_mean_expr.shape}')
print(f'NTC mean protein DSB shape: {ntc_mean_protein.shape}')

# --- Step 2: Gradient function using NTC mean ---
def compute_protein_gradients_ntc_mean(model, enformer_emb, enf_idx,
                                       ntc_mean_rna, ntc_mean_prot,
                                       target_cdt_idx, n_proteins, device):
    """Compute d(Protein_pred[j]) / d(rna_input[target_gene]) using NTC mean.

    NO perturbation data is used. Only:
      - DNA: target gene's Enformer embedding
      - RNA: NTC mean expression (unperturbed baseline)
      - Protein: NTC mean DSB (unperturbed baseline)
    """
    dna_input = torch.from_numpy(
        enformer_emb[enf_idx:enf_idx+1].astype(np.float32)).to(device)
    rna_input = torch.from_numpy(
        ntc_mean_rna.reshape(1, -1).astype(np.float32)).to(device)
    rna_input.requires_grad_(True)
    prot_input = torch.from_numpy(
        ntc_mean_prot.reshape(1, -1).astype(np.float32)).to(device)

    rna_pred, prot_pred = model(dna_input, rna_input, prot_input)

    grads_per_protein = np.zeros(n_proteins)
    for j in range(n_proteins):
        model.zero_grad()
        if rna_input.grad is not None:
            rna_input.grad = None
        prot_pred[0, j].backward(retain_graph=True)
        grads_per_protein[j] = rna_input.grad[0, target_cdt_idx].item()

    return grads_per_protein


# --- Step 3: Run NTC-mean gradient for CD52 ---
CD52_CDT_IDX = gene_to_cdt_idx.get('CD52')
CD52_TARGET_IDX = gene_name_to_target_idx.get('CD52')
enf_idx = target_to_enformer[CD52_TARGET_IDX]

print('\nComputing CD52 -> all proteins gradient using NTC mean only...')
print('(No CD52 perturbation data used)')

cd52_ntc_grads = compute_protein_gradients_ntc_mean(
    model, enformer_emb, enf_idx,
    ntc_mean_expr, ntc_mean_protein,
    CD52_CDT_IDX, n_proteins, device)

# --- Step 4: Compare with original (perturbation-cell-based) gradients ---
print('\n' + '=' * 60)
print('COMPARISON: NTC Mean vs Perturbation Cell Gradients')
print('=' * 60)

from scipy.stats import spearmanr
r_all, p_all = pearsonr(cd52_ntc_grads, cd52_protein_grads)
rho_all, p_rho_all = spearmanr(cd52_ntc_grads, cd52_protein_grads)
print(f'\nAll {n_proteins} proteins:')
print(f'  Pearson r = {r_all:.4f} (p = {p_all:.2e})')
print(f'  Spearman rho = {rho_all:.4f} (p = {p_rho_all:.2e})')

r_expr, p_expr = pearsonr(cd52_ntc_grads[expressed_mask], cd52_protein_grads[expressed_mask])
rho_expr, p_rho_expr = spearmanr(cd52_ntc_grads[expressed_mask], cd52_protein_grads[expressed_mask])
print(f'\n{n_expressed} expressed proteins:')
print(f'  Pearson r = {r_expr:.4f} (p = {p_expr:.2e})')
print(f'  Spearman rho = {rho_expr:.4f} (p = {p_rho_expr:.2e})')

# Direction agreement
ntc_dir = np.sign(cd52_ntc_grads)
orig_dir = np.sign(cd52_protein_grads)
dir_agree_all = (ntc_dir == orig_dir).sum()
dir_agree_expr = (ntc_dir[expressed_mask] == orig_dir[expressed_mask]).sum()
print(f'\nDirection agreement:')
print(f'  All: {dir_agree_all}/{n_proteins} ({100*dir_agree_all/n_proteins:.1f}%)')
print(f'  Expressed: {dir_agree_expr}/{n_expressed} ({100*dir_agree_expr/n_expressed:.1f}%)')

# --- Step 5: Top 20 comparison ---
print('\n' + '=' * 60)
print('Top 20 Proteins: NTC Mean vs Perturbation Cell')
print('=' * 60)
abs_ntc = np.abs(cd52_ntc_grads)
ranked_ntc = np.argsort(abs_ntc)[::-1]

for rank, idx in enumerate(ranked_ntc[:20]):
    prot_label = adt_to_protein.get(protein_names[idx], protein_names[idx])
    gene_label = adt_to_gene.get(protein_names[idx], '')
    ntc_val = cd52_ntc_grads[idx]
    orig_val = cd52_protein_grads[idx]
    match = 'OK' if np.sign(ntc_val) == np.sign(orig_val) else 'DIFF'
    print(f'  {rank+1:2d}. {prot_label:<25s} NTC={ntc_val:+.6f}  Orig={orig_val:+.6f}  {match}')

# --- Step 6: Scatter plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(cd52_protein_grads, cd52_ntc_grads, alpha=0.5, s=20, c='gray')
axes[0].scatter(cd52_protein_grads[expressed_mask], cd52_ntc_grads[expressed_mask],
                alpha=0.7, s=30, c='blue', label=f'Expressed ({n_expressed})')
lims = [min(cd52_protein_grads.min(), cd52_ntc_grads.min()),
        max(cd52_protein_grads.max(), cd52_ntc_grads.max())]
axes[0].plot(lims, lims, 'r--', alpha=0.5, label='y=x')
axes[0].set_xlabel('Gradient (perturbation cells)')
axes[0].set_ylabel('Gradient (NTC mean)')
axes[0].set_title(f'CD52 Protein Gradients\nr={r_all:.3f} (all), r={r_expr:.3f} (expressed)')
axes[0].legend()

axes[1].scatter(cd52_protein_grads[expressed_mask], cd52_ntc_grads[expressed_mask],
                alpha=0.7, s=40, c='blue')
abs_expr = np.abs(cd52_ntc_grads[expressed_mask])
top5 = np.argsort(abs_expr)[::-1][:5]
expr_idx = np.where(expressed_mask)[0]
for ti in top5:
    ri = expr_idx[ti]
    label = adt_to_protein.get(protein_names[ri], protein_names[ri])
    axes[1].annotate(label, (cd52_protein_grads[ri], cd52_ntc_grads[ri]),
                     fontsize=8, alpha=0.8)
lims_e = [cd52_protein_grads[expressed_mask].min(), cd52_protein_grads[expressed_mask].max()]
axes[1].plot(lims_e, lims_e, 'r--', alpha=0.5)
axes[1].set_xlabel('Gradient (perturbation cells)')
axes[1].set_ylabel('Gradient (NTC mean)')
axes[1].set_title(f'Expressed Proteins (n={n_expressed})\nr={r_expr:.3f}')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gradient_ntc_vs_perturbation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')


In [ ]:
# ============================================================
# Save Results
# ============================================================

# TRS results
trs_df.to_csv(OUTPUT_BASE / 'trs_results.csv', index=False)

# Side effect predictions
if cd52_protein_grads is not None:
    side_effect_df.to_csv(OUTPUT_BASE / 'cd52_side_effect_predictions.csv', index=False)

# Resistance candidates
if resistance_df is not None:
    resistance_df.to_csv(OUTPUT_BASE / 'cd52_resistance_candidates.csv', index=False)

# Gradient summary
grad_summary = {g: {
    'grad_vs_exp_r': float(r['grad_vs_exp_r']),
    'trs': float(r['trs']),
    'cos_sim': float(r['cos_sim']),
    'n_cells': r['n_cells'],
} for g, r in gradient_results.items()}

with open(OUTPUT_BASE / 'gradient_trs_summary.json', 'w') as f:
    json.dump(grad_summary, f, indent=2)

print("Saved:")
print(f"  {OUTPUT_BASE / 'trs_results.csv'}")
print(f"  {OUTPUT_BASE / 'cd52_side_effect_predictions.csv'}")
print(f"  {OUTPUT_BASE / 'cd52_resistance_candidates.csv'}")
print(f"  {OUTPUT_BASE / 'gradient_trs_summary.json'}")
print("\nNB2 complete.")